In [0]:
from pyspark.sql import functions as F

df = spark.table("silver_jobs")
job_skills = spark.table("job_skills")

In [0]:
duplicate_jobs = (
    df.groupBy("source", "source_url")
      .count()
      .filter(F.col("count") > 1)
      .count()
)

total_rows = df.count()

null_source_url = (
    df.filter(
        F.col("source_url").isNull()
        | (F.trim(F.col("source_url")) == "")
    ).count()
)

null_title = (
    df.filter(
        F.col("title").isNull()
        | (F.trim(F.col("title")) == "")
    ).count()
)

invalid_salary_range = (
    df.filter(
        F.col("salary_min").isNotNull()
        & F.col("salary_max").isNotNull()
        & (F.col("salary_min") > F.col("salary_max"))
    ).count()
)

negative_salary = (
    df.filter(
        (F.col("salary_min") < 0)
        | (F.col("salary_max") < 0)
    ).count()
)

unsupported_salary = (
    df.filter(
        F.col("salary_type") == "unsupported"
    ).count()
)

unmapped_it_title = (
    df.filter(
        (F.col("is_it_job") == True)
        & (F.col("job_title_group") == "Other IT")
    ).count()
)

In [0]:
it_jobs = (
    df.filter(F.col("is_it_job") == True)
      .select("source", "source_url")
      .distinct()
)

jobs_with_skills = (
    job_skills
    .select("source", "source_url")
    .distinct()
)

missing_skill = (
    it_jobs
    .join(
        jobs_with_skills,
        ["source", "source_url"],
        "left_anti"
    )
    .count()
)

In [0]:
dq_results = [
    ("duplicate_job", duplicate_jobs, "ERROR"),
    ("null_source_url", null_source_url, "ERROR"),
    ("null_title", null_title, "ERROR"),
    ("invalid_salary_range", invalid_salary_range, "ERROR"),
    ("negative_salary", negative_salary, "ERROR"),

    ("unsupported_salary", unsupported_salary, "WARNING"),
    ("unmapped_it_title", unmapped_it_title, "WARNING"),
    ("missing_skill", missing_skill, "WARNING")
]

dq_df = spark.createDataFrame(
    dq_results,
    ["check_name", "failed_rows", "severity"]
)

dq_df = dq_df.withColumn(
    "status",
    F.when(
        F.col("failed_rows") == 0,
        F.lit("PASS")
    ).otherwise(
        F.when(
            F.col("severity") == "ERROR",
            F.lit("FAIL")
        ).otherwise(F.lit("WARNING"))
    )
)

display(dq_df)

check_name,failed_rows,severity,status
duplicate_job,0,ERROR,PASS
null_source_url,0,ERROR,PASS
null_title,0,ERROR,PASS
invalid_salary_range,0,ERROR,PASS
negative_salary,0,ERROR,PASS
unsupported_salary,0,WARNING,PASS
unmapped_it_title,8,WARNING,WARNING
missing_skill,26,WARNING,WARNING


In [0]:
error_count = (
    dq_df
    .filter(
        (F.col("severity") == "ERROR")
        & (F.col("status") == "FAIL")
    )
    .count()
)

if error_count > 0:
    raise Exception(
        f"Data Quality FAILED: {error_count} critical checks failed."
    )

print("Data Quality PASSED.")

Data Quality PASSED.
